# Steam Game Price Tracker #


This AI tool is designed to help users to track steam game price

**In order to run this tool, you need to have a developer API from this site, "isthereanydeal.com" and store the API key into the .env file**




In [ ]:
import os
import sys
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import sqlite3
from datetime import datetime, timedelta



In [ ]:
# --- Configuration ---
load_dotenv(override=True)
ITAD_API_KEY = os.getenv('ITAD-API-KEY')


if not ITAD_API_KEY: 
    print("ITAD_API_KEY was not found in the env file. Please add the API key first and try again")
    sys.exit("Error: Condition was not met. Exiting script.")


In [ ]:
country_code_options = ['US', 'JP', 'GB', 'CA', 'AU', 'DE', 'FR']
selected_country = 'US'

def change_country_code(code):
    global selected_country
    selected_country = code
    print(f"country code updated: {selected_country}")
    return

In [ ]:
# Initialize your OpenAI client configured for your Llama 3.1 endpoint
# (e.g., Together AI, Groq, Fireworks, or a local vLLM/Ollama instance)

MODEL_API_KEY = os.getenv('MODEL_API_KEY')

if not MODEL_API_KEY: 
    print("MODEL_API_KEY was not found in the env file. Please add the API key first and try again")
    sys.exit("Error: Condition was not met. Exiting script.")

client = OpenAI(
    base_url="http://localhost:11434/v1", 
    api_key=MODEL_API_KEY
)

In [ ]:
DB_FILE = "steam_tracker.db"


In [ ]:
system_prompt = """You are a helpful gaming deal assistant that helps users track Steam game prices using an intelligent local caching system.

CRITICAL RULES FOR TOOL USE:
1. ONLY call the 'fetch_steam_game_price_history' tool if the user explicitly mentions a specific video game title they want to check.
2. If the user says hello, asks general questions, or gives a vague query without a clear game title, DO NOT call any tools. Instead, reply politely and ask them what specific game they want to look up.
3. Never guess, hallucinate, or make up a game title if one isn't clearly provided in the chat history.

HOW TO INTEGRATE AND REASON WITH THE TOOL DATA:
1. The tool automatically checks a local SQL 'game_library' database first. If the record is less than 24 hours old, it returns cached data to save API calls. Otherwise, it updates with live data.
2. When summarizing the deal, do not mention database implementation details (like SQLite or tables) to the end user unless they explicitly ask how the application works. 
3. Focus your summary on:
   - The current deal price vs the regular price (calculate the discount percentage if not already clear).
   - How close the current deal is to the historical lowest price (calculate the exact dollar difference to show value).
   - Provide a clear, analytical conclusion on whether they should buy now or wait for a better sale.
"""

In [ ]:
def fetch_steam_game_price_history(game_title: str, country: str = "US") -> str:
    headers = {"ITAD-API-Key": ITAD_API_KEY}
    
    # -------------------------------------------------------------------------
    # STEP 1: Search the ITAD ID first to get the standardized identifier
    # -------------------------------------------------------------------------
    search_url = "https://api.isthereanydeal.com/games/search/v1"
    try:
        print(f'calling /search API for {game_title, country}')
        search_res = requests.get(search_url, headers=headers, params={"title": game_title})
        search_res.raise_for_status()
        search_data = search_res.json()
        if not search_data:
            return json.dumps({"error": f"No games found matching '{game_title}'."})
        
        game_id = search_data[0]["id"]
        official_title = search_data[0]["title"]
        print(f"found official title: {official_title}")
    except Exception as e:
        print(f"Search route failed: {str(e)}")
        return json.dumps({"error": f"Search route failed: {str(e)}"})

    # -------------------------------------------------------------------------
    # STEP 2: Check the Local Cache
    # -------------------------------------------------------------------------
    print('Look up the title in the database...')
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT current_deal, historical_lowest, last_search_date FROM game_library WHERE id = ? AND country_code = ?",
            (game_id, country)
        )
        row = cursor.fetchone()
        
    if row:
        print(f'game found in the db')
        cached_current, cached_lowest, last_search_str = row
        last_search_time = datetime.fromisoformat(last_search_str)
        
        # Check if the cache record is fresh (less than 24 hours old)
        if datetime.now() - last_search_time < timedelta(hours=24):
            print(f"⚡ Cache Hit! Loading '{official_title}' ({country}) data locally.")
            return json.dumps({
                "title": official_title,
                "current_deal": json.loads(cached_current),
                "historical_lowest": json.loads(cached_lowest),
                "cached": True
            })

    # -------------------------------------------------------------------------
    # STEP 3: Cache Miss or Stale Data -> Query the API
    # -------------------------------------------------------------------------
    print(f"🌐 Cache Miss/Stale! Fetching fresh API data for '{official_title}' ({country}).")
    overview_url = "https://api.isthereanydeal.com/games/overview/v2"
    try:
        overview_res = requests.post(overview_url, headers=headers, params={"country": country}, json=[game_id])
        overview_res.raise_for_status()
        overview_data = overview_res.json()
        
        prices_list = overview_data.get("prices", [])
        if not prices_list:
            return json.dumps({"error": f"No pricing details available for {official_title}."})
            
        game_deal_info = prices_list[0]
        current_deal = game_deal_info.get("current")
        historical_lowest = game_deal_info.get("lowest")
        now_str = datetime.now().isoformat()
        
        # -------------------------------------------------------------------------
        # STEP 4: Insert or Update the Database
        # -------------------------------------------------------------------------
        with sqlite3.connect(DB_FILE) as conn:
            print('Inserting or Updating data in the table')
            cursor = conn.cursor()
            cursor.execute("""
                INSERT INTO game_library (id, title, country_code, current_deal, historical_lowest, last_search_date)
                VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(id, country_code) DO UPDATE SET
                    current_deal = excluded.current_deal,
                    historical_lowest = excluded.historical_lowest,
                    last_search_date = excluded.last_search_date;
            """, (game_id, official_title, country, json.dumps(current_deal), json.dumps(historical_lowest), now_str))
            conn.commit()
            print("Done inserting/updating data")

        return json.dumps({
            "title": official_title,
            "current_deal": current_deal,
            "historical_lowest": historical_lowest,
            "cached": False
        })
        
    except Exception as e:
        return json.dumps({"error": f"Overview route failed: {str(e)}"})


In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "fetch_steam_game_price_history",
            "description": "Searches for a video game and retrieves its current deals, regular pricing, and historical all-time lowest price in the selected country.",
            "parameters": {
                "type": "object",
                "properties": {
                    "game_title": {
                        "type": "string",
                        "description": "The name of the video game (e.g., 'Cyberpunk 2077', 'Hades')."
                    },
                    "country": {
                        "type": "string",
                        "description": "Two letter country code (EX: US, GB, CA)"
                    }
                },
                "required": ["game_title", "country"]
            }
        }
    }
]

In [ ]:
# =====================================================================
# 3. THE AGENT EXECUTION LOOP
# =====================================================================
def ask_llama(user_query: str):
    print(f"\nUser: {user_query}")
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
    
    # First call: Send prompt and tools schema to Llama 3.1
    response = client.chat.completions.create(
        model="llama3.1", # Ensure this matches your exact hosted model name string
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Check if Llama 3.1 wants to run our API tool
    if tool_calls:
        print("🤖 Llama 3.1 decided to call the price tracker tool...")
        messages.append(response_message) # Append Llama's tool call intent to context
        
        for tool_call in tool_calls:
            if tool_call.function.name == "fetch_steam_game_price_history":
                # Extract argument from Llama's structured response
                function_args = json.loads(tool_call.function.arguments)
                extracted_title = function_args.get("game_title")
                
                # Execute our local Python code
                tool_output = fetch_steam_game_price_history(game_title=extracted_title, country=selected_country)
                
                # Send the clean JSON payload back into the chat sequence
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": "fetch_steam_game_price_history",
                    "content": tool_output
                })
        
        # Second call: Feed the API data back to Llama 3.1 so it can summarize
        second_response = client.chat.completions.create(
            model="llama3.1",
            messages=messages
        )
        print(f"🤖 Llama 3.1: {second_response.choices[0].message.content}")
    else:
        # If no tool was required (e.g., user just said "hello")
        print(f"🤖 Llama 3.1: {response_message.content}")



In [ ]:
# --- Run a Test ---
if __name__ == "__main__":
    ask_llama("Is Clair Obscur: Expedition 33 worth buying right now or should I wait?")

# Gradio UI

In [ ]:
def message_text(content):
    if isinstance(content, str):
        return content
    return "".join(part.get("text", "") for part in content if part.get("type") == "text")

def chat(message, history):
    if not message.strip():
        return "", history

    # Gradio 6 returns each message's content as a list of parts, e.g. [{"type": "text", "text": "..."}]
    history = [{"role":h["role"], "content":message_text(h["content"])} for h in history]
    history.append({"role":"user", "content":message})
    messages = [{"role": "system", "content": system_prompt}] + history
    # First call: Send prompt and tools schema to Llama 3.1
    response = client.chat.completions.create(
        model="llama3.1",
        messages=messages,
        tools=tools_schema,
    )
    
    response_message = response.choices[0].message
    print(response_message)
    tool_calls = response_message.tool_calls
    
    # Check if Llama 3.1 wants to run our API tool
    if tool_calls:
        print("🤖 Llama 3.1 decided to call the price tracker tool...")
        messages.append(response_message) # Append Llama's tool call intent to context
        
        for tool_call in tool_calls:
            if tool_call.function.name == "fetch_steam_game_price_history":
                # Extract argument from Llama's structured response
                function_args = json.loads(tool_call.function.arguments)
                extracted_title = function_args.get("game_title")
                
                # Execute our local Python code
                tool_output = fetch_steam_game_price_history(game_title=extracted_title, country=selected_country)
                
                # Send the clean JSON payload back into the chat sequence
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": "fetch_steam_game_price_history",
                    "content": tool_output
                })
        
        # Second call: Feed the API data back to Llama 3.1 so it can summarize
        second_response = client.chat.completions.create(
            model="llama3.1",
            messages=messages
        )
        print(f"🤖 Llama 3.1: {second_response.choices[0].message.content}")
        bot_reply = second_response.choices[0].message.content
    else:
        print(f"🤖 Llama 3.1: {response_message.content}")
        bot_reply = response.choices[0].message.content

    history.append({"role":"assistant", "content": bot_reply})
    return "", history
    

In [ ]:
initial_greeting = [
    {
        "role": "assistant", 
        "content": "👋 Hello! I am your Steam Price Tracking Assistant. Provide me with a game title, and I will instantly check its current deals and compare them to its historical all-time low to let you know if it's the right time to buy! Which game are we looking up today?"
    }
]

with gr.Blocks(title="Game Price Tracker AI") as ui:
    gr.Markdown('# Steam Game Price tracker AI Agent')
    with gr.Row():
        with gr.Column(scale=3, min_width=500):
            chatbot = gr.Chatbot(value=initial_greeting, height=500)
            with gr.Row():
                txt_input = gr.Textbox(show_label=False, placeholder="Ask about a game", scale=4)
                submit_btn = gr.Button('Send', variant="primary", scale=1)
        with gr.Column(scale=1):
            gr.Markdown('### Control & Watchlist Panel')
            with gr.Accordion("⚙️ Advanced Options", open=True):
                with gr.Column():
                    country_code_dropdown = gr.Dropdown(
                        choices=country_code_options, 
                        label="Select Country", 
                        value=country_code_options[0],
                        interactive=True  # Explicitly forces interactivity
                    )

    country_code_dropdown.change(fn=change_country_code, inputs=country_code_dropdown)
    txt_input.submit(fn=chat, inputs=[txt_input, chatbot], outputs=[txt_input, chatbot])
    submit_btn.click(
        fn=chat, 
        inputs=[txt_input, chatbot], 
        outputs=[txt_input, chatbot]
    )

ui.launch()
        